# Models Point-E And Shape-E
## Google Colab — Runtime Actual + A100


---
## Célula 1 — Instalar Point-E

In [ ]:
!pip install git+https://github.com/openai/point-e.git


In [ ]:
import os

# Limpar montagem antiga que ficou presa
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    # Tenta desmontar primeiro
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except:
        pass
    # Se ainda tem arquivos, remove o diretório
    import shutil
    shutil.rmtree('/content/drive', ignore_errors=True)

from google.colab import drive
drive.mount('/content/drive')

# Configurar cache no Drive
PROJECT_DIR = '/content/drive/MyDrive/Ufma/ECP/CG'
os.environ['XDG_CACHE_HOME'] = PROJECT_DIR
os.makedirs(PROJECT_DIR + '/cache_modelos', exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"✓ Diretório: {os.getcwd()}")

---
## Configurar modelo

In [ ]:
# Inferência Point-E
import torch
from PIL import Image
from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.plotting import plot_point_cloud

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Modelo base (image → point cloud baixa resolução)
print("Carregando modelo base...")
base_name = 'base1B'  # ou 'base40M' para mais rápido/menos qualidade
base_model = model_from_config(MODEL_CONFIGS[base_name], device)
base_model.eval()
base_diffusion = diffusion_from_config(DIFFUSION_CONFIGS[base_name])
base_model.load_state_dict(load_checkpoint(base_name, device))

In [ ]:
# 2. Upsampler (point cloud baixa → alta resolução)
print("Carregando upsampler...")
upsampler_model = model_from_config(MODEL_CONFIGS['upsample'], device)
upsampler_model.eval()
upsampler_diffusion = diffusion_from_config(DIFFUSION_CONFIGS['upsample'])
upsampler_model.load_state_dict(load_checkpoint('upsample', device))

# 3. Sampler
sampler = PointCloudSampler(
    device=device,
    models=[base_model, upsampler_model],
    diffusions=[base_diffusion, upsampler_diffusion],
    num_points=[1024, 4096 - 1024],
    aux_channels=['R', 'G', 'B'],
    guidance_scale=[3.0, 0.0],
    model_kwargs_key_filter=('images', ''),
)

In [ ]:
print(f"✓ Diretório: {os.getcwd()}")

pasta = '/content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs'
print(os.listdir(pasta))

In [ ]:
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.pc_to_mesh import marching_cubes_mesh
def criar_modelos_ply(pc,name):
  # 6. (Opcional) Converter point cloud para mesh com modelo SDF
    print("Carregando modelo SDF...")
    sdf_name = 'sdf'
    sdf_model = model_from_config(MODEL_CONFIGS[sdf_name], device)
    sdf_model.eval()
    sdf_model.load_state_dict(load_checkpoint(sdf_name, device))

    mesh = marching_cubes_mesh(
        pc=pc,
        model=sdf_model,
        batch_size=4096,
        grid_size=128,
        progress=True,)
    # Salvar mesh
    with open(name, 'wb') as f:
        mesh.write_ply(f)
    print("Mesh salvo em "+name)

In [ ]:
name  = os.path.splitext(os.path.basename("/content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/busto-edited.png"))[0]

print(name)

In [ ]:
import glob, subprocess, sys


imgs = sorted(glob.glob(pasta+"/*.*"))

print(f"Imagens encontradas: {len(imgs)}")
for img in imgs:
    print(f"  - {img}")

for img in imgs:
    print(f"\n{'='*50}")
    print(f"Processando: {img}")
    print(f"{'='*50}", flush=True)
    path=img
    img = Image.open(path)  # Sua imagem de entrada
    samples = None
    for x in sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(images=[img])):
        samples = x

    # 5. Visualizar point cloud
    pc = sampler.output_to_point_clouds(samples)[0]
    fig = plot_point_cloud(pc, grid_size=3, fixed_bounds=((-0.75, -0.75, -0.75),(0.75, 0.75, 0.75)))

    name = name  = os.path.splitext(os.path.basename(path))[0] + '_point_e.ply'
    criar_modelos_ply(pc,name)



print(f"\n✓ Processamento concluído!")

##Gerar e visualizar o point cloud

In [ ]:
# 4. Gerar point cloud a partir de imagem
# import glob, subprocess, sys

# imgs = sorted(glob.glob("./inputs/edited/*.*"))
path=pasta
img = Image.open(path)  # Sua imagem de entrada
samples = None
for x in sampler.sample_batch_progressive(batch_size=1, model_kwargs=dict(images=[img])):
    samples = x

# 5. Visualizar point cloud
pc = sampler.output_to_point_clouds(samples)[0]
fig = plot_point_cloud(pc, grid_size=3, fixed_bounds=((-0.75, -0.75, -0.75),(0.75, 0.75, 0.75)))


In [ ]:
print(pcs)

##Conversão para mesh

In [ ]:
# 6. (Opcional) Converter point cloud para mesh com modelo SDF
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.pc_to_mesh import marching_cubes_mesh

print("Carregando modelo SDF...")
sdf_name = 'sdf'
sdf_model = model_from_config(MODEL_CONFIGS[sdf_name], device)
sdf_model.eval()
sdf_model.load_state_dict(load_checkpoint(sdf_name, device))

mesh = marching_cubes_mesh(
    pc=pc,
    model=sdf_model,
    batch_size=4096,
    grid_size=128,
    progress=True,
)

In [ ]:
# Salvar mesh
with open('output_point_e.ply', 'wb') as f:
    mesh.write_ply(f)
print("Mesh salvo em output_point_e.ply")

In [ ]:
!pip install trimesh

In [ ]:

import trimesh
import plotly.graph_objects as go
import numpy as np
imgs = sorted(glob.glob(PROJECT_DIR+"/modelos/*.*"))

print(f"Imagens encontradas: {len(imgs)}")
for img in imgs:
    print(f"  - {img}")
    mesh_path=img
    mesh = trimesh.load_mesh(mesh_path)
    vertices = mesh.vertices
    faces = mesh.faces

    # Extrair cores dos vértices (RGBA -> RGB normalizado)
    if mesh.visual.vertex_colors is not None:
        colors = mesh.visual.vertex_colors[:, :3]  # RGB
        # Converter para string de cor por vértice
        vertex_colors = ['rgb({},{},{})'.format(r, g, b) for r, g, b in colors]

        fig = go.Figure(data=[go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            vertexcolor=colors,
            opacity=1.0
        )])
    else:
        fig = go.Figure(data=[go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            color='lightblue', opacity=0.9
        )])

    fig.update_layout(scene=dict(aspectmode='data'))
    fig.show()